# LLM orchestration testing

Run the cells in order to inspect every IPO extraction step. Ollama cells are opt-in so you can inspect all local processing without sending model requests.

In [1]:
import logging
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = project_root.parent

llm_orchestration_directory = project_root / 'src' / 'llm_orchestration'
web_crawling_output_directory = project_root / 'src' / 'web_crawling' / 'output'
sys.path.insert(0, str(llm_orchestration_directory))

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(name)s | %(message)s')
available_crawler_output_files = sorted(web_crawling_output_directory.glob('*.json'))
if not available_crawler_output_files:
    raise FileNotFoundError(f'No crawler JSON files found in {web_crawling_output_directory}')

crawler_output_file = available_crawler_output_files[0]
print(f'Project root: {project_root}')
print(f'Crawler output file: {crawler_output_file.name}')

Project root: /Users/ahmad/Documents/Python DS/ipo_oracle
Crawler output file: research_results.json


In [2]:
# Set working directory
import os


os.chdir(project_root)

In [3]:
# Load environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

OLLAMA_SERVER_URL = os.getenv('OLLAMA_SERVER_URL')
OLLAMA_MODEL_NAME = os.getenv('OLLAMA_MODEL_NAME')
OLLAMA_REQUEST_TIMEOUT_SECONDS = int(os.getenv('OLLAMA_REQUEST_TIMEOUT_SECONDS'))
MAX_WEBPAGE_CONTENT_LENGTH = int(os.getenv('MAX_WEBPAGE_CONTENT_LENGTH'))

print({
    'ollama_server_url': OLLAMA_SERVER_URL,
    'ollama_model_name': OLLAMA_MODEL_NAME,
    'request_timeout_seconds': OLLAMA_REQUEST_TIMEOUT_SECONDS,
    'maximum_webpage_content_length': MAX_WEBPAGE_CONTENT_LENGTH,
})


{'ollama_server_url': 'http://localhost:11434', 'ollama_model_name': 'llama3.2', 'request_timeout_seconds': 120, 'maximum_webpage_content_length': 15000}


In [9]:
from src.helper.file_import import import_json
webpages_data = import_json(crawler_output_file)['crawled_pages']

In [10]:
type(webpages_data)

list

In [11]:
from extraction.crawler_json_loader import load_crawled_webpages_from_json

crawled_webpages = load_crawled_webpages_from_json(str(crawler_output_file))
first_crawled_webpage = crawled_webpages[0]

print(f'Loaded webpages: {len(crawled_webpages)}')
print(f'Title: {first_crawled_webpage.webpage_title}')
print(f'URL: {first_crawled_webpage.source_url}')
print(f'Content length: {len(first_crawled_webpage.webpage_content):,} characters')
print(first_crawled_webpage.webpage_content[:500])

2026-08-29 12:54:22,433 | INFO | extraction.crawler_json_loader | Loaded 10 crawled webpages from /Users/ahmad/Documents/Python DS/ipo_oracle/src/web_crawling/output/research_results.json


Loaded webpages: 10
Title: IPO GMP Today, Live IPO Grey Market Premium Price - IPO Watch
URL: https://ipowatch.in/ipo-grey-market-premium-latest-ipo-gmp/
Content length: 77,160 characters
[Skip to content]( [ ![IPO Watch Logo]( ]( [ ]( [ Ads ]( [ Apply IPO ]( * [ IPO GMP ]( * IPOs Close IPOs Open IPOs * All IPOs * [ Upcoming IPO ]( * [ Review by Dilip Davda ]( * [ IPO Review ]( * [ IPO Forms ]( * [ IPO Subscription Status ]( * [ IPO Allotment Status ]( * [ IPO Listing ]( * Current IPOs * [ Annu Projects ]( * [ Sumax Engineering ]( * [ Madhur Knit Crafts ]( * [ ABH Healthcare ]( * [ Symbiotec Pharmalab ]( * [ Hy-Tech Engineers ]( * [ Skyways Air ]( * Mainboard IPOs * [ Upcoming Ma


In [14]:
crawled_webpages[0].webpage_content

"[Skip to content]( [ ![IPO Watch Logo]( ]( [ ]( [ Ads ]( [ Apply IPO ]( * [ IPO GMP ]( * IPOs Close IPOs Open IPOs * All IPOs * [ Upcoming IPO ]( * [ Review by Dilip Davda ]( * [ IPO Review ]( * [ IPO Forms ]( * [ IPO Subscription Status ]( * [ IPO Allotment Status ]( * [ IPO Listing ]( * Current IPOs * [ Annu Projects ]( * [ Sumax Engineering ]( * [ Madhur Knit Crafts ]( * [ ABH Healthcare ]( * [ Symbiotec Pharmalab ]( * [ Hy-Tech Engineers ]( * [ Skyways Air ]( * Mainboard IPOs * [ Upcoming Mainboard IPO ]( * [ Rays of Belief ]( * [ Priority Jewels ]( * [ Lumino Industries ]( * SME IPOs * [ Upcoming SME IPO ]( * [ Shanti Inorganics ]( * [ Paluck Technologies ]( * [ Complete Sports ]( * [ Kwick Forensic Solutions ]( * Other Investments Close Other Investments Open Other Investments * NCD Issues * [ Upcoming NCD ]( * [ Kosamattam Finance ]( * [ KLM Axiva Finvest ]( * [ Indel Money ]( * [ ICL Fincorp ]( * [ Paisalo Digital ]( * Rights Issue (RI) * [ Upcoming Rights Issue ]( * [ Viceroy

In [10]:
from extraction.ipo_extraction_prompt import (
    IPO_INFORMATION_EXTRACTION_SYSTEM_PROMPT,
    create_ipo_information_extraction_prompt,
)

webpage_content_for_prompt = first_crawled_webpage.webpage_content[:MAX_WEBPAGE_CONTENT_LENGTH]
ipo_extraction_prompt = create_ipo_information_extraction_prompt(
    webpage_content=webpage_content_for_prompt,
    source_url=first_crawled_webpage.source_url,
)

print(f'Prompt length: {len(ipo_extraction_prompt):,} characters')
print(ipo_extraction_prompt[:2_000])

2026-08-29 12:37:52,901 | INFO | extraction.ipo_extraction_prompt | Starting IPO information extraction prompt creation
2026-08-29 12:37:52,902 | INFO | extraction.ipo_extraction_prompt | Adding crawled webpage content from https://ipowatch.in/ipo-grey-market-premium-latest-ipo-gmp/ to the prompt


Prompt length: 16,406 characters


You are an IPO information extraction system.

Your task is to extract IPO information from crawled webpage content.

Extract ONLY information explicitly present in the webpage.

Never guess.
Never infer missing information.
Never calculate values unless explicitly requested.

Return ONLY valid JSON.

Use exactly this JSON structure:

{
    "company_name": null,
    "ipo_opening_date": null,
    "ipo_closing_date": null,
    "grey_market_premium": null,
    "issue_price": null,
    "lot_size": null,
    "source_url": null
}

FIELD DEFINITIONS:

company_name:
The company name associated with the IPO.

ipo_opening_date:
The date on which IPO subscription opens.

ipo_closing_date:
The date on which IPO subscription closes.

grey_market_premium:
The Grey Market Premium (GMP) mentioned on the webpage.

Preserve the GMP value as stated.
For example:
"₹25"
"Rs 25"
"25"

Do not calculate GMP.

issue_price:
The IPO issue price or price band.

Examples:
"₹95"
"

In [11]:
from extraction.ipo_data_validator import normalize_ipo_lot_size, validate_and_normalize_ipo_information

sample_unvalidated_ipo_information = {
    'company_name': 'Example Industries Limited',
    'ipo_opening_date': '2026-08-28',
    'ipo_closing_date': '2026-09-01',
    'grey_market_premium': '₹25',
    'issue_price': '₹95 - ₹100',
    'lot_size': '1,200',
    'source_url': first_crawled_webpage.source_url,
    'unexpected_field': 'removed during validation',
}

validated_sample_ipo_information = validate_and_normalize_ipo_information(sample_unvalidated_ipo_information)
print(validated_sample_ipo_information)
print(normalize_ipo_lot_size('1,500 shares'))

2026-08-29 12:37:52,916 | INFO | extraction.ipo_data_validator | Starting IPO information validation and normalization
2026-08-29 12:37:52,917 | INFO | extraction.ipo_data_validator | Starting IPO lot-size normalization
2026-08-29 12:37:52,917 | INFO | extraction.ipo_data_validator | Lot size normalized successfully
2026-08-29 12:37:52,918 | INFO | extraction.ipo_data_validator | Validated IPO information contains 7 populated fields
2026-08-29 12:37:52,918 | INFO | extraction.ipo_data_validator | IPO information validation and normalization completed
2026-08-29 12:37:52,919 | INFO | extraction.ipo_data_validator | Starting IPO lot-size normalization
2026-08-29 12:37:52,920 | WARNING | extraction.ipo_data_validator | Could not convert the supplied lot size into an integer


{'company_name': 'Example Industries Limited', 'ipo_opening_date': '2026-08-28', 'ipo_closing_date': '2026-09-01', 'grey_market_premium': '₹25', 'issue_price': '₹95 - ₹100', 'lot_size': 1200, 'source_url': 'https://ipowatch.in/ipo-grey-market-premium-latest-ipo-gmp/'}
None


## Ollama tests

Start Ollama and ensure the configured model is installed before enabling these cells; for example, run `ollama serve` and `ollama pull llama3.2`.

In [15]:
from extraction.ollama_llm_client import OllamaLanguageModelClient

run_ollama_request = True  # Set to True to call the local Ollama server.
ollama_client = OllamaLanguageModelClient()

if run_ollama_request:
    raw_ollama_response = ollama_client.generate_text(ipo_extraction_prompt)
    print(raw_ollama_response)
else:
    print('Ollama request skipped. Set run_ollama_request = True to test the client.')

ModuleNotFoundError: No module named 'configuration'

In [13]:
if run_ollama_request:
    structured_ollama_response = ollama_client.generate_structured_json(ipo_extraction_prompt)
    print(structured_ollama_response)
else:
    structured_ollama_response = None
    print('Structured JSON test skipped because Ollama requests are disabled.')

Structured JSON test skipped because Ollama requests are disabled.


In [14]:
from extraction.ipo_information_extractor import IPOInformationExtractor

if run_ollama_request:
    ipo_information_extractor = IPOInformationExtractor(ollama_client)
    extracted_ipo_record = ipo_information_extractor.extract_ipo_information_from_webpage(first_crawled_webpage)
    print(extracted_ipo_record)
else:
    extracted_ipo_record = validated_sample_ipo_information
    print('Single-page extraction skipped; using the validated sample for formatter tests.')

Single-page extraction skipped; using the validated sample for formatter tests.


In [15]:
from ipo_table_formatter import (
    convert_ipo_records_to_dataframe,
    convert_ipo_records_to_markdown_table,
    save_ipo_records_to_csv,
)

ipo_records_for_formatting = [extracted_ipo_record]
ipo_information_dataframe = convert_ipo_records_to_dataframe(ipo_records_for_formatting)
ipo_information_markdown_table = convert_ipo_records_to_markdown_table(ipo_records_for_formatting)
notebook_csv_output_file = project_root / 'src' / 'llm_orchestration' / 'output' / 'notebook_ipo_information.csv'
save_ipo_records_to_csv(ipo_records_for_formatting, str(notebook_csv_output_file))

display(ipo_information_dataframe)
print(ipo_information_markdown_table)
print(f'Saved formatter test output to: {notebook_csv_output_file}')

2026-08-29 12:37:53,416 | INFO | ipo_table_formatter | Starting IPO record dataframe creation
2026-08-29 12:37:53,416 | INFO | ipo_table_formatter | Converting 1 extracted IPO records into a dataframe
2026-08-29 12:37:53,422 | INFO | ipo_table_formatter | IPO record dataframe creation completed
2026-08-29 12:37:53,422 | INFO | ipo_table_formatter | Starting IPO markdown-table creation
2026-08-29 12:37:53,422 | INFO | ipo_table_formatter | Starting IPO record dataframe creation
2026-08-29 12:37:53,423 | INFO | ipo_table_formatter | Converting 1 extracted IPO records into a dataframe
2026-08-29 12:37:53,423 | INFO | ipo_table_formatter | IPO record dataframe creation completed
2026-08-29 12:37:53,423 | INFO | ipo_table_formatter | Converting IPO dataframe into a markdown table
2026-08-29 12:37:53,429 | INFO | ipo_table_formatter | IPO markdown-table creation completed
2026-08-29 12:37:53,430 | INFO | ipo_table_formatter | Starting IPO record CSV export
2026-08-29 12:37:53,430 | INFO | ip

,company_name,ipo_opening_date,ipo_closing_date,grey_market_premium,issue_price,lot_size,source_url
0,Example Industries Limited,2026-08-28,2026-09-01,₹25,₹95 - ₹100,1200,https://ipowatch.in/ipo-grey-market-premium-la...


| company_name               | ipo_opening_date   | ipo_closing_date   | grey_market_premium   | issue_price   |   lot_size | source_url                                                  |
|:---------------------------|:-------------------|:-------------------|:----------------------|:--------------|-----------:|:------------------------------------------------------------|
| Example Industries Limited | 2026-08-28         | 2026-09-01         | ₹25                   | ₹95 - ₹100    |       1200 | https://ipowatch.in/ipo-grey-market-premium-latest-ipo-gmp/ |
Saved formatter test output to: /Users/ahmad/Documents/Python DS/ipo_oracle/src/llm_orchestration/output/notebook_ipo_information.csv


In [16]:
from main import run_ipo_information_extraction_pipeline
import main as pipeline_module

run_full_pipeline = True  # Set to True to extract IPO information from every crawled webpage.
pipeline_module.CRAWLER_OUTPUT_JSON_FILE = str(crawler_output_file)
pipeline_module.IPO_OUTPUT_CSV_FILE = str(project_root / 'src' / 'llm_orchestration' / 'output' / 'ipo_information.csv')

if run_full_pipeline:
    full_pipeline_dataframe = run_ipo_information_extraction_pipeline()
    display(full_pipeline_dataframe)
else:
    print('Full pipeline skipped. Set run_full_pipeline = True to run all webpage extractions.')

2026-08-29 12:37:53,451 | INFO | main | Starting IPO information extraction pipeline
2026-08-29 12:37:53,451 | INFO | main | Loading crawled webpage records from /Users/ahmad/Documents/Python DS/ipo_oracle/src/web_crawling/output/research_results.json
2026-08-29 12:37:53,453 | INFO | extraction.crawler_json_loader | Loaded 10 crawled webpages from /Users/ahmad/Documents/Python DS/ipo_oracle/src/web_crawling/output/research_results.json
2026-08-29 12:37:53,453 | INFO | main | Loaded 10 crawled webpage records
2026-08-29 12:37:53,454 | INFO | main | Creating the IPO information extractor
2026-08-29 12:37:53,454 | INFO | extraction.ipo_information_extractor | Starting IPO information extractor initialization
2026-08-29 12:37:53,454 | INFO | extraction.ollama_llm_client | Starting Ollama language-model client initialization
2026-08-29 12:37:53,454 | INFO | extraction.ollama_llm_client | Ollama language-model client initialized for model llama3.2 at http://localhost:11434
2026-08-29 12:37:5

Loading crawled webpages...
Loaded 10 crawled webpages.
Extracting IPO information with Ollama...


2026-08-29 12:38:10,105 | INFO | extraction.ollama_llm_client | Ollama text-generation request completed successfully
2026-08-29 12:38:10,116 | INFO | extraction.ollama_llm_client | Returning generated text from Ollama
2026-08-29 12:38:10,117 | INFO | extraction.ollama_llm_client | Parsing Ollama response as JSON
2026-08-29 12:38:10,118 | INFO | extraction.ollama_llm_client | Structured JSON generation completed successfully
2026-08-29 12:38:10,118 | INFO | extraction.ipo_information_extractor | Validating the extracted IPO information
2026-08-29 12:38:10,118 | INFO | extraction.ipo_data_validator | Starting IPO information validation and normalization
2026-08-29 12:38:10,119 | INFO | extraction.ipo_data_validator | Starting IPO lot-size normalization
2026-08-29 12:38:10,119 | INFO | extraction.ipo_data_validator | Lot size is missing; returning no normalized value
2026-08-29 12:38:10,119 | INFO | extraction.ipo_data_validator | Validated IPO information contains 0 populated fields
202

Successfully extracted: https://ipowatch.in/ipo-grey-market-premium-latest-ipo-gmp/


2026-08-29 12:38:27,126 | INFO | extraction.ollama_llm_client | Ollama text-generation request completed successfully
2026-08-29 12:38:27,137 | INFO | extraction.ollama_llm_client | Returning generated text from Ollama
2026-08-29 12:38:27,140 | INFO | extraction.ollama_llm_client | Parsing Ollama response as JSON
2026-08-29 12:38:27,141 | INFO | extraction.ollama_llm_client | Structured JSON generation completed successfully
2026-08-29 12:38:27,141 | INFO | extraction.ipo_information_extractor | Validating the extracted IPO information
2026-08-29 12:38:27,142 | INFO | extraction.ipo_data_validator | Starting IPO information validation and normalization
2026-08-29 12:38:27,142 | INFO | extraction.ipo_data_validator | Starting IPO lot-size normalization
2026-08-29 12:38:27,142 | INFO | extraction.ipo_data_validator | Lot size is missing; returning no normalized value
2026-08-29 12:38:27,142 | INFO | extraction.ipo_data_validator | Validated IPO information contains 1 populated fields
202

Successfully extracted: https://www.ipo360.in/gmp


2026-08-29 12:38:55,727 | INFO | extraction.ollama_llm_client | Ollama text-generation request completed successfully
2026-08-29 12:38:55,739 | INFO | extraction.ollama_llm_client | Returning generated text from Ollama
2026-08-29 12:38:55,742 | INFO | extraction.ollama_llm_client | Parsing Ollama response as JSON
2026-08-29 12:38:55,742 | INFO | extraction.ollama_llm_client | Structured JSON generation completed successfully
2026-08-29 12:38:55,742 | INFO | extraction.ipo_information_extractor | Validating the extracted IPO information
2026-08-29 12:38:55,743 | INFO | extraction.ipo_data_validator | Starting IPO information validation and normalization
2026-08-29 12:38:55,743 | INFO | extraction.ipo_data_validator | Starting IPO lot-size normalization
2026-08-29 12:38:55,743 | INFO | extraction.ipo_data_validator | Lot size is missing; returning no normalized value
2026-08-29 12:38:55,743 | INFO | extraction.ipo_data_validator | Validated IPO information contains 0 populated fields
202

Successfully extracted: https://www.ipomarket.in/gmp


KeyboardInterrupt: 

In [ ]:
import ollama


def generate_llama_response(prompt: str,
                            webpage_content: str,
                            model: str = "iodose/nuextract-v1.5") -> str:
    """
    Generate a response from the local model using Ollama.

    Args:
        prompt: The prompt to send to the model.
        model: The model to use for generation.

    Returns:
        The model-generated response as a string.
    """
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt,
            },
            {
                "role": "assistant",
                "content": webpage_content,
            }
        ],
    )

    llm_response = response["message"]["content"]
    return f"Model used {model}\n{llm_response}"

In [ ]:
prompt = """
Extract all the companies going for IPO with their GMP, opening date, closing date, issue price, and lot size from the provided text.
"""

response = generate_llama_response(prompt,crawled_webpages[1].webpage_content)

print(response)

2026-08-29 09:21:00,513 | INFO | httpx | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


Model used iodose/nuextract-v1.5
Extracted information from the provided text:

1. Company: Deepa Jewellers
   IPO Issue Price: ₹168 - ₹177
   Open Date: 1 Sep
   Close Date: 3 Sep
   Listing Date: 8 Sep
   GMP (Grey Market Premium): ₹48
   Expected listing gain: 27.12%

2. Company: Rays of Belief
   IPO Issue Price: ₹227 - ₹239
   Open Date: 1 Sep
   Close Date: 3 Sep
   Listing Date: 8 Sep
   GMP (Grey Market Premium): ₹31
   Expected listing gain: 12.97%

3. Company: Farm Peace
   IPO Issue Price: ₹59
   Open Date: 1 Sep
   Close Date: 3 Sep
   Listing Date: 8 Sep
   GMP (Grey Market Premium): ₹0
   Expected listing gain: 0.00%

4. Company: Ashutosh Fibre
   IPO Issue Price: ₹87 - ₹92
   Open Date: 31 Aug-2 Sep
   Close Date: 7 Sep
   Listing Date: 7 Sep
   GMP (Grey Market Premium): ₹38
   Expected listing gain: 41.30%

5. Company: Phychem
   IPO Issue Price: ₹51 - ₹54
   Open Date: 31 Aug-2 Sep
   Close Date: 7 Sep
   Listing Date: 7 Sep
   GMP (Grey Market Premium): ₹3
   Expecte

In [ ]:
for pages in crawled_webpages:
    print("Length of characters in each page:",len(pages.webpage_content))

Length of characters in each page: 77160
Length of characters in each page: 9874
Length of characters in each page: 17803
Length of characters in each page: 13201
Length of characters in each page: 27451
Length of characters in each page: 16126
Length of characters in each page: 21263
Length of characters in each page: 39862
Length of characters in each page: 27043
Length of characters in each page: 7181


In [ ]:
print(crawled_webpages[1].webpage_content)

[ ![IPO360]( ]( [ ]( Dashboard [Live IPO Dashboard]( [Live GMP Tracker]( [Live Subscription]( [Live Allotment Status]( [ Unlisted ]( [ Forum ]( Reports [Upcoming IPOs]( [Past IPOs]( [BRLM]( Tools [IPO Fund Allocator]( [ Partner Pro ]( [Log in]( Quick Menu [Log in]( #### Live Market [ Live IPOs ]( [ GMP Tracker ]( [ Subscriptions ]( [ Allotment ]( #### Research & Tools [ Upcoming ]( [ Past IPOs ]( [ BRLM ]( [ Allocator ]( [ Invest in Unlisted Shares ]( [ Partner Pro ]( 1. [Home]( 2. [IPO Dashboard]( 3. Live IPO GMP # Live IPO GMP (Grey Market Premium) Today Real-time **Grey Market Premium** for all upcoming and ongoing **Mainboard** and **SME IPOs**. Real time **GMP** for all ongoing **Mainboard** and **SME IPOs** Live Data: Active GMPs [Live IPO Dashboard]( [Live Subscription]( [Allotment Status]( ## Live GMP Table All Open Upcoming Closed Live Data: Active GMPs Search company name All Mainboard SME List of ongoing and upcoming Mainboard and SME IPOs with their issue prices, live Grey 